In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name in {'Baseline_Training', 'CTC_models', 'PhoneticFeatures_Training'}:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from project_paths import (
    BASELINE_MODELS_DIR,
    CTC_MODELS_DIR,
    DEFAULT_BASELINE_MODEL,
    DEFAULT_CTC_ATTENTION_MODEL,
    DEFAULT_CTC_MODEL,
    DEFAULT_PHONETIC_MODEL_2CLASS,
    DEFAULT_PHONETIC_MODEL_3CLASS,
    EXAMPLE_EMBEDDINGS,
    EXAMPLE_PHONEMES,
    EXAMPLE_SEG_B2,
    EXAMPLE_SEG_B4,
    EXAMPLE_WAV,
    PHONEME_CHOICES_SUMMARY,
    PHONETIC_MODELS_DIR,
    TABLES_DIR,
)


# Оценкак качества транскрипции по трипленым предсказаниям

In [ ]:
import torch
import os
from tqdm import tqdm

from Decoding_tools import get_phrase
from GetData import load_phrases
from main import get_phrase_prediciton
from ToIPA import corpres2ipa
import panphon.distance

from Levenshtein import distance

import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, precision_recall_curve,average_precision_score
from sklearn.preprocessing import label_binarize
from itertools import permutations
from Decoding_tools import viterbi_decode, viterbi_lookahead, write_to_seg


def cer(true, pred, ignore_stress=True):
    tr = []
    pr = []
    if ignore_stress:
        for t in true:
            if t[0] in 'aeouyi':
                tr.append(t[0])
            else:
                tr.append(t)
        for p in pred:
            if p[0] in 'aeouyi':
                pr.append(p[0])
            else:
                pr.append(p)
    else:
        tr = true
        pr = pred
    

    lev_dist = distance(tr, pr)
    cer_score = lev_dist / len(true) if true else 0.0
    return lev_dist, cer_score

def pfer(true, pred):
    return 0

def normalize_prob_dist(prob_dist, eps=1e-8):
    out = []
    for frame in prob_dist:
        frame_dict = {ph: max(p, eps) for ph, p in frame}
        out.append(frame_dict)
    return out




In [ ]:
def compute_error_rate(prob_distributions, true_labelling_list, ignore_stress=True,
                                                                lambda_penalty=4.0,
                                                                lookahead=4,):

    probability_dist = {
            "start": normalize_prob_dist(prob_distributions["start"]),
            "center": normalize_prob_dist(prob_distributions["centre"]),
            "end": normalize_prob_dist(prob_distributions["end"]),
        }

    phonemes, phonemes_no_rep, boundaries = viterbi_decode(probability_dist)
    phonemes_la, phonemes_no_rep_la, boundaries_la = viterbi_lookahead(probability_dist, 
                                                                lambda_penalty=4.0,
                                                                lookahead=4,)

    basic_levinstein, basic_cer = cer(true_labelling_list, phonemes_no_rep, ignore_stress=ignore_stress)
    la_levinstein, la_cer = cer(true_labelling_list, phonemes_no_rep_la, ignore_stress=ignore_stress)

    basic_pfer = pfer(true_labelling_list, phonemes_no_rep)
    la_pfer = pfer(true_labelling_list, phonemes_no_rep_la)

    return {
        'probability_distribution': probability_dist,
        'viterbi_decoded_phonemes': phonemes,
        'viterbi_decoded_phonemes_no_repetition': phonemes_no_rep,
        'viterbi_decoded_phonemes_la': phonemes_la,
        'viterbi_decoded_phonemes_no_repetition_la': phonemes_la,
        'metrics': {
            'basic_levinstein': basic_levinstein,
            'basic_cer': basic_cer,
            'la_levinstein': la_levinstein,
            'la_cer': la_cer,
            'basic_pfer': basic_pfer,
            'la_pfer': la_pfer,
        }
    }

## Данные

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.cuda.empty_cache()
root_dir = r'\\nid-tts-02\mnt\hot_store\trainee_common\ananeva\CORPRES_embs_no_averaging'
save_path = str(PROJECT_ROOT)
speakers = ['Vladimir', 'Maria', 'Mikhail', 'Anna', 'Galina', 'Victoria', 'Petr', 'Alexander']
phoneme_list_full = ['a0', 'a1', 'a2', 'a4', 'b', "b'", 'c', 'ch', 'ch_', 'd', "d'", 'e0', 'e1', 'e2', 'e4', 'f', "f'", 'g', "g'", 'h', "h'", 'i0', 'i1', 'i2', 'i4', 'j', 'jr', 'jl', 'ji4', 'k', "k'", 'l', "l'", 'm', "m'", 'n', "n'", 'o0', 'o1', 'o2', 'o4', 'p', "p'", 'r', "r'", 's', "s'", 'sc', 'sh', 't', "t'", 'u0', 'u1', 'u2', 'u4', 'v', "v'", 'y0', 'y1', 'y2', 'y4', 'z', "z'", 'zh', "zh'", 'C', 'CH', 'H', 'SC']

In [ ]:
model_id = 'RGCAIQYZHB' #'VACXUXVEXO'
phoneme_list = ['a0', 'a4', 'b', "b'", 'c', 'ch', 'd', "d'", 'e0', 'f', "f'", 'g', 'h', 'i0','i4', 'j', 'k', "k'", 'l', "l'", 'm', "m'", 'n', "n'", 'o0', 'p', "p'", 'r', "r'", 's', "s'", 'sh', 't', "t'", 'u0', 'v', "v'", 'y0', 'z', "z'", 'zh', 'sil']
# phoneme_list = ['a0', 'a4', 'b', 'd','e0', 'f',  'g', 'h', 'i0','i4', 'j', 'k','l', 'm',  'n', 'o0', 'p', 'r', 's', 'sh', 't', 'u0', 'v',  'y0', 'z', 'zh', 'sil']
# phoneme_list = ['a0', 't', "i0", "t'", "k", 's', "s'" ]  
phoneme_list = ['a0', 'a4', 'b', "b'", 'c', 'ch', 'd', "d'", 'e0', 'f', "f'", 'g', 'h', 'i0','i4', 'j', 'k', "k'", 'l', "l'", 'm', "m'", 'n', "n'", 'o0', 'p', "p'", 'r', "r'", 's', "s'", 'sh', 't', "t'", 'u0', 'v', "v'", 'y0', 'z', "z'", 'zh', 'sil', 'a1', 'i1','u1', 'y1', ]
# интересующие фонемы (без позиций)
samples_of_phrases = 1000              # сколько примеров на диктора
train_speakers = ['Vladimir', 'Maria', 'Mikhail', 'Anna']
test_speakers = ['Galina', 'Victoria', 'Petr', 'Alexander']
    

Подгрузка модели

In [ ]:
model = torch.load(os.path.join(save_path, model_id + '_model.pth'), map_location='cpu')
model.to(device)

Данные для оценки

In [ ]:
embedding_sequences, phoneme_targets, info = load_phrases(
    root_dir=root_dir,
    speakers=test_speakers,
    max_sequences_per_speaker=1000)

In [ ]:
for l in phoneme_targets:
    print(list(l.keys())[0])
    print(get_phrase(list(l.values())[0]))

## Оценка качества

### PER

In [ ]:
prob_distributions = {'start': [], 'centre': [], 'end': []}
y_pred = {'start': [], 'centre': [], 'end': []}
result = []


 С ударением

In [ ]:
overall_cer_la = 0
overall_cer = 0
predictions = []
for phrase, targets in tqdm(zip(embedding_sequences, phoneme_targets)):
    y_pred,prob_distributions, result = get_phrase_prediciton(phrase, model, phoneme_list, device)
    predictions.append(prob_distributions)
    
    true_phrase, true_list = get_phrase(list(targets.values())[0])        
    metrics = compute_error_rate(prob_distributions, true_list, ignore_stress=False)
    overall_cer_la += metrics['metrics']['la_cer']
    overall_cer += metrics['metrics']['basic_cer']
    

print(overall_cer_la/len(phoneme_targets))
print(overall_cer/len(phoneme_targets))


Без ударения

In [ ]:
overall_cer_la = 0
overall_cer = 0
for prob_distributions, targets in tqdm(zip(predictions, phoneme_targets)):
    true_phrase, true_list = get_phrase(list(targets.values())[0])        
    metrics = compute_error_rate(prob_distributions, true_list, ignore_stress=True)
    overall_cer_la += metrics['metrics']['la_cer']
    overall_cer += metrics['metrics']['basic_cer']
    

print(overall_cer_la/len(phoneme_targets))
print(overall_cer/len(phoneme_targets))


### Подбор пареметров для lookahead

In [ ]:
overall_cer_la = 0
overall_cer = 0
for prob_distributions, targets in tqdm(zip(predictions, phoneme_targets)):
    true_phrase, true_list = get_phrase(list(targets.values())[0])        
    metrics = compute_error_rate(prob_distributions, true_list, ignore_stress=True,
                                                                lambda_penalty=4.0,
                                                                lookahead=4, )
    overall_cer_la += metrics['metrics']['la_cer']
    overall_cer += metrics['metrics']['basic_cer']
    

print(overall_cer_la/len(phoneme_targets))
print(overall_cer/len(phoneme_targets))


In [ ]:
overall_cer_la = 0
overall_cer = 0
for prob_distributions, targets in tqdm(zip(predictions, phoneme_targets)):
    true_phrase, true_list = get_phrase(list(targets.values())[0])        
    metrics = compute_error_rate(prob_distributions, true_list, ignore_stress=True,
                                                                lambda_penalty=4.0,
                                                                lookahead=5, )
    overall_cer_la += metrics['metrics']['la_cer']
    overall_cer += metrics['metrics']['basic_cer']
    

print(overall_cer_la/len(phoneme_targets))
print(overall_cer/len(phoneme_targets))


In [ ]:
overall_cer_la = 0
overall_cer = 0
for prob_distributions, targets in tqdm(zip(predictions, phoneme_targets)):
    true_phrase, true_list = get_phrase(list(targets.values())[0])        
    metrics = compute_error_rate(prob_distributions, true_list, ignore_stress=True,
                                                                lambda_penalty=5.0,
                                                                lookahead=5)
    overall_cer_la += metrics['metrics']['la_cer']
    overall_cer += metrics['metrics']['basic_cer']
    

print(overall_cer_la/len(phoneme_targets))
print(overall_cer/len(phoneme_targets))


In [ ]:
overall_cer_la = 0
overall_cer = 0
for prob_distributions, targets in tqdm(zip(predictions, phoneme_targets)):
    true_phrase, true_list = get_phrase(list(targets.values())[0])        
    metrics = compute_error_rate(prob_distributions, true_list, ignore_stress=True,
                                                                lambda_penalty=5.0,
                                                                lookahead=4,)
    overall_cer_la += metrics['metrics']['la_cer']
    overall_cer += metrics['metrics']['basic_cer']
    

print(overall_cer_la/len(phoneme_targets))
print(overall_cer/len(phoneme_targets))


In [ ]:
overall_cer_la = 0
overall_cer = 0
for prob_distributions, targets in tqdm(zip(predictions, phoneme_targets)):
    true_phrase, true_list = get_phrase(list(targets.values())[0])        
    metrics = compute_error_rate(prob_distributions, true_list, ignore_stress=True,
                                                                lambda_penalty=6.0,
                                                                lookahead=4,)
    overall_cer_la += metrics['metrics']['la_cer']
    overall_cer += metrics['metrics']['basic_cer']
    

print(overall_cer_la/len(phoneme_targets))
print(overall_cer/len(phoneme_targets))


## Без декодирования 

In [ ]:
overall_cer_nostress = 0
overall_cer = 0
count = 0
for prob_distributions, targets in tqdm(zip(predictions, phoneme_targets)):
    while count < 10:

        true_phrase, true_list = get_phrase(list(targets.values())[0])  
        probability_dist = {
                "start": normalize_prob_dist(prob_distributions["start"]),
                "center": normalize_prob_dist(prob_distributions["centre"]),
                "end": normalize_prob_dist(prob_distributions["end"]),
            }
    
        phonemes, phonemes_no_rep, boundaries = viterbi_decode(probability_dist)
       
        pred = phonemes
        true = list(targets.values())[0]
        lev_dist_nostress, cer_score_nostress = cer(true, pred, ignore_stress=True)
        lev_dist, cer_score = cer(true, pred, ignore_stress=False)
        
        overall_cer_nostress += cer_score_nostress
        overall_cer += cer_score
        
        count += 1
        print('_'.join(pred),
              '_'.join(true),
              cer_score_nostress,
              cer_score
              )
        
        
    

    
print(overall_cer_nostress/len(phoneme_targets))
print(overall_cer/len(phoneme_targets))


### Простое декодирование

In [ ]:
overall_cer_nostress = 0
overall_cer = 0
count = 0
for prob_distributions, targets in tqdm(zip(predictions, phoneme_targets)):

        true_phrase, true_list = get_phrase(list(targets.values())[0])  
        probability_dist = {
                "start": normalize_prob_dist(prob_distributions["start"]),
                "center": normalize_prob_dist(prob_distributions["centre"]),
                "end": normalize_prob_dist(prob_distributions["end"]),
            }
    
        phonemes, phonemes_no_rep, boundaries = viterbi_decode(probability_dist)
       
        phrase, pred = get_phrase(phonemes_no_rep)
        true = list(targets.values())[0]
        lev_dist_nostress, cer_score_nostress = cer(true, pred, ignore_stress=True)
        lev_dist, cer_score = cer(true, pred, ignore_stress=False)
        
        overall_cer_nostress += cer_score_nostress
        overall_cer += cer_score
    
print(overall_cer_nostress/len(phoneme_targets))
print(overall_cer/len(phoneme_targets))


## PFER

### Без декодирования

In [ ]:
import panphon.distance

dst = panphon.distance.Distance()
overall_pfer = 0
overall_pfer_la = 0
n_features=24
for prob_distributions, targets in tqdm(zip(predictions, phoneme_targets)):

    true_phrase, true_list = get_phrase(list(targets.values())[0])  
    probability_dist = {
            "start": normalize_prob_dist(prob_distributions["start"]),
            "center": normalize_prob_dist(prob_distributions["centre"]),
            "end": normalize_prob_dist(prob_distributions["end"]),
        }

    phonemes, phonemes_no_rep, boundaries = viterbi_decode(probability_dist)
    phonemes_la, phonemes_no_rep_la, boundaries_la = viterbi_lookahead(probability_dist)
    
    
    ipa_pred = corpres2ipa(phonemes)
    ipa_pred_la = corpres2ipa(phonemes_la)
    ipa_true = corpres2ipa(list(targets.values())[0])
    true_len = len([s for s in ipa_true.split(' ') if s!=''])
    
    
    dist = dst.feature_edit_distance(ipa_pred, ipa_true)
    overall_pfer += dist/true_len
    dist_la = dst.feature_edit_distance(ipa_pred_la, ipa_true)
    overall_pfer_la += dist_la/true_len


    

print(overall_pfer_la/len(phoneme_targets))
print(overall_pfer/len(phoneme_targets))


In [ ]:


dst  = panphon.distance.Distance()
overall_pfer = 0
overall_pfer_la = 0
overall_pfer_full = 0
n_features=24
for prob_distributions, targets in tqdm(zip(predictions, phoneme_targets)):
    true_phrase, true_list = get_phrase(list(targets.values())[0])  
    probability_dist = {
            "start": normalize_prob_dist(prob_distributions["start"]),
            "center": normalize_prob_dist(prob_distributions["centre"]),
            "end": normalize_prob_dist(prob_distributions["end"]),
        }

    phonemes, phonemes_no_rep, boundaries = viterbi_decode(probability_dist)
    phonemes_la, phonemes_no_rep_la, boundaries_la = viterbi_lookahead(probability_dist)
    
    ipa_pred = corpres2ipa(phonemes_no_rep)
    ipa_pred_la = corpres2ipa(phonemes_no_rep_la)
    ipa_true = corpres2ipa(true_list)
    true_len = len(true_list)
    
    ipa_pred_full = corpres2ipa(phonemes)
    ipa_true_full = corpres2ipa(list(targets.values())[0])
    true_len_full = len(list(targets.values())[0])

    
    
    dist = dst.feature_edit_distance(ipa_pred, ipa_true)
    overall_pfer += dist/true_len
    dist_la = dst.feature_edit_distance(ipa_pred_la, ipa_true)
    overall_pfer_la += dist_la/true_len
    dist_full = dst.feature_edit_distance(ipa_pred_full, ipa_true_full)
    overall_pfer_full += dist_full/true_len_full

    
print(f'PFER with basic Viterbi decoding:  {overall_pfer/len(phoneme_targets)}')
print(f'PFER with lookahead Viterbi decoding:  {overall_pfer_la/len(phoneme_targets)}')
print(f'PFER without decoding:  {overall_pfer_full/len(phoneme_targets)}')



In [ ]:
ipa_pred, ipa_pred_la, ipa_true 


In [ ]:
true_list

In [ ]:
dist = dst.feature_edit_distance(ipa_pred, ipa_true)

In [ ]:
dist/24

In [ ]:
dist = dst.feature_edit_distance('k a l', 'p o ')

In [ ]:
dist

In [ ]:
dist1 = dst.feature_edit_distance('k', 'p')
dist1

In [ ]:
dist2 = dst.feature_edit_distance('a', 'o')
dist2

In [ ]:
dist3 = dst.feature_edit_distance('l', '')
dist3

In [ ]:
(dist1+dist2+dist3)